In [ ]:
# PyTorch 기본 라이브러리 임포트
import torch
# 신경망 구성을 위한 nn 모듈 임포트
import torch.nn as nn
# 모델 최적화를 위한 옵티마이저 임포트
import torch.optim as optim
# 데이터 로딩 및 배치 처리를 위한 유틸리티 임포트
from torch.utils.data import DataLoader, TensorDataset
# MNIST 데이터셋과 데이터 변환 도구 임포트
from torchvision import datasets, transforms

# Apple Silicon GPU 사용 가능 여부 확인
if torch.backends.mps.is_available():
    # MPS (Metal Performance Shaders) 디바이스 설정
    device = torch.device("mps")
    print(f"Using Apple device : device={device}")
else:
    # GPU가 없는 경우 CPU 사용
    device = torch.device("cpu")
    print(f"Using CPU device : device={device}")    

# 데이터 전처리를 위한 변환 파이프라인 정의
transform = transforms.Compose([
    # 이미지를 PyTorch 텐서로 변환 (0-255 값을 0-1로 정규화)
    transforms.ToTensor(),
    # MNIST 데이터셋에 대한 정규화 (평균과 표준편차 적용)
    transforms.Normalize((0.1307,), (0.3081,))
])

# 학습용 MNIST 데이터셋 로드 (60,000개의 학습 이미지)
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
# 테스트용 MNIST 데이터셋 로드 (10,000개의 테스트 이미지)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)   

# 미니배치 크기 설정 (한 번에 처리할 이미지 수)
batch_size = 64
# 학습 데이터 로더 생성 (데이터 셔플링 활성화)
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
# 테스트 데이터 로더 생성 (데이터 셔플링 비활성화)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)    

# 숫자 분류를 위한 신경망 모델 클래스 정의
class SimpleClassifier(nn.Module):
    def __init__(self):
        # 부모 클래스 초기화
        super(SimpleClassifier, self).__init__()
        # 2D 이미지를 1D 벡터로 평탄화하는 레이어
        self.flatten = nn.Flatten()
        # 순차적 레이어 구성
        self.layer_stack = nn.Sequential(
            # 입력층(784=28x28) -> 은닉층1(512)
            nn.Linear(28*28, 512),
            # 비선형성 추가를 위한 ReLU 활성화 함수
            nn.ReLU(),
            # 은닉층1(512) -> 은닉층2(512)
            nn.Linear(512, 512),
            # 두 번째 ReLU 활성화 함수
            nn.ReLU(),
            # 출력층(512 -> 10, 각 숫자 클래스에 대한 점수)
            nn.Linear(512, 10)
        )

    # 순전파 함수 정의 (입력 -> 출력)
    def forward(self, x):
        # 입력 이미지를 1차원으로 평탄화
        x = self.flatten(x)
        # 레이어를 통과시켜 최종 출력(로짓) 계산
        logits = self.layer_stack(x)
        return logits

# 모델 인스턴스 생성 및 지정된 디바이스로 이동
model = SimpleClassifier().to(device)
# 손실 함수로 크로스 엔트로피 손실 사용 (다중 클래스 분류에 적합)
criterion = nn.CrossEntropyLoss()
# Adam 옵티마이저 설정 (학습률 0.001로 설정)
optimizer = optim.Adam(model.parameters(), lr=0.001)
# 전체 데이터셋 반복 횟수 설정
num_epochs = 5

# 모델 학습 루프 시작
for epoch in range(num_epochs):
    # 모델을 학습 모드로 설정
    model.train()
    # 에포크별 누적 손실값 초기화
    running_loss = 0.0
    # 미니배치 단위로 데이터 처리
    for images, labels in train_loader:
        # 데이터를 지정된 디바이스로 이동
        images, labels = images.to(device), labels.to(device)

        # 모델을 통한 순전파 수행
        outputs = model(images)
        # 예측값과 실제값 간의 손실 계산
        loss = criterion(outputs, labels)

        # 이전 그래디언트 초기화
        optimizer.zero_grad()
        # 역전파를 통한 그래디언트 계산
        loss.backward()
        # 가중치 업데이트
        optimizer.step()

        # 배치별 손실값 누적 (배치 크기를 고려하여)
        running_loss += loss.item() * images.size(0)

    # 에포크의 평균 손실값 계산 및 출력
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')    

# 학습된 모델의 성능 평가
model.eval()
# 그래디언트 계산 비활성화 (메모리 절약)
with torch.no_grad():
    # 정확도 계산을 위한 변수 초기화
    correct = 0
    total = 0
    # 테스트 데이터 처리
    for images, labels in test_loader:
        # 테스트 데이터를 디바이스로 이동
        images, labels = images.to(device), labels.to(device)
        # 모델을 통한 예측 수행
        outputs = model(images)
        # 가장 높은 확률을 가진 클래스 선택
        _, predicted = torch.max(outputs.data, 1)
        # 전체 테스트 샘플 수 누적
        total += labels.size(0)
        # 정확히 예측한 샘플 수 누적
        correct += (predicted == labels).sum().item()

    # 최종 정확도 계산 및 출력 (백분율)
    print(f'Accuracy of the model on the test images: {100 * correct / total:.2f}%')    

# 학습된 모델의 가중치를 파일로 저장 (나중에 재사용 가능)
torch.save(model.state_dict(), 'simple_classifier.pth')

Using Apple device : device=mps
Epoch [1/5], Loss: 0.1988
Epoch [1/5], Loss: 0.1988
Epoch [2/5], Loss: 0.0866
Epoch [2/5], Loss: 0.0866
Epoch [3/5], Loss: 0.0622
Epoch [3/5], Loss: 0.0622
Epoch [4/5], Loss: 0.0481
Epoch [4/5], Loss: 0.0481
Epoch [5/5], Loss: 0.0388
Epoch [5/5], Loss: 0.0388
Accuracy of the model on the test images: 97.87%
Accuracy of the model on the test images: 97.87%
